# Part 2.2 — Logistic Regression
**Owner: Student 3** · Dataset: 6-class Emotion Classification.

**Goal:** Implement binary **bigram** features and **multiclass softmax logistic regression from scratch using NumPy**, train with shuffled mini-batches/SGD, and compare against scikit-learn `LogisticRegression` and `SGDClassifier`.

**Shared split:** 80% train / 10% validation / 10% test, `SEED=42`, stratified. Build your bigram vocabulary using **training data only**. Do not touch test labels while tuning models.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# Support launching Jupyter from the repo root or notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression, SGDClassifier
from utils.data_loader import load_emotion_splits
from utils.preprocessing import tokenize
from utils.evaluation import sklearn_evaluation

## 2. Load and preprocess the shared 80/10/10 dataset
Student 2 uses **exactly the same** `load_emotion_splits(seed=42)` function.

In [ ]:
data = load_emotion_splits(seed=SEED)
print('Train / validation / test:', len(data.train), len(data.val), len(data.test))
print('Class names:', list(enumerate(data.class_names)))
assert (len(data.train), len(data.val), len(data.test)) == (12800, 1600, 1600)
assert data.num_classes == 6

train_tokens = [tokenize(text) for text in data.train.texts]
val_tokens = [tokenize(text) for text in data.val.texts]
test_tokens = [tokenize(text) for text in data.test.texts]

print('Example:', train_tokens[0], 'label:', data.class_names[data.train.labels[0]])

## 3. Binary bigram feature extraction (from scratch)
The lab specifically asks for a **sparse binary vector**, one dimension per unique bigram.

**Recommended NumPy-only scratch representation:** each document is an array of **unique active bigram indices** (not a huge dense 0/1 array). This is an equivalent sparse representation of a binary feature vector. For example, if the train-only vocabulary is `{('i','feel'):0, ('very','happy'):1}`, the sentence `i feel very happy i feel` has indices `[0,1]` — each bigram exists once, regardless of frequency.

**TODO:** Build the bigram vocabulary from **training** tokens and transform train/val/test with the *same* mapping. Ignore unseen bigrams in validation/test. Your logistic regression implementation must use NumPy, **not** scikit-learn estimators or vectorizers.

In [ ]:
def build_bigram_vocabulary(tokenized_train: list[list[str]]) -> dict[tuple[str, str], int]:
    # TODO: extract consecutive (word_1, word_2) pairs from training texts only.
    # Create a stable mapping: bigram tuple -> integer feature ID.
    raise NotImplementedError('Student 3: implement bigram vocabulary')


def transform_binary_bigrams(
    tokenized_texts: list[list[str]], vocabulary: dict[tuple[str, str], int]
) -> list[np.ndarray]:
    # TODO: return one sorted np.ndarray of UNIQUE active feature IDs per text.
    # The array is the sparse representation of a binary feature vector.
    # Never add unseen bigrams to vocabulary here.
    raise NotImplementedError('Student 3: implement sparse binary bigram features')

### Quick feature tests (run after completing both functions)
Expected: repeated bigrams are **binary** and unseen bigrams add **zero** active indices.

In [ ]:
# TODO: Uncomment when the two functions above work.
# example_vocab = build_bigram_vocabulary([['i', 'feel', 'happy'], ['i', 'feel', 'happy']])
# assert len(example_vocab) == 2
# example_rows = transform_binary_bigrams([['i', 'feel', 'happy', 'i', 'feel', 'happy']], example_vocab)
# assert len(example_rows[0]) == 2
# assert len(transform_binary_bigrams([['unseen', 'pair']], example_vocab)[0]) == 0
# bigram_vocab = build_bigram_vocabulary(train_tokens)
# X_train_rows = transform_binary_bigrams(train_tokens, bigram_vocab)
# X_val_rows = transform_binary_bigrams(val_tokens, bigram_vocab)
# X_test_rows = transform_binary_bigrams(test_tokens, bigram_vocab)
# print('Bigram vocabulary size:', len(bigram_vocab))

## 4. NumPy-only multiclass logistic regression (softmax)
Implement six-way classification. Store `W` with shape `(num_features, num_classes)` and `b` with shape `(num_classes,)`.

**Hints:** For a sample with active bigram indices `ids`, its unnormalized scores can be computed using only the activated rows: `W[ids].sum(axis=0) + b`. Use a numerically stable softmax by subtracting the maximum logit, compute cross-entropy, gradients, and update parameters. **Shuffle every epoch**, use mini-batches/SGD, and record a loss curve. Avoid allocating a dense matrix of size `12800 × vocabulary_size`.

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, num_features: int, num_classes: int, *,
                 learning_rate: float = 0.05, epochs: int = 10,
                 batch_size: int = 64, seed: int = SEED):
        self.num_features = num_features
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.seed = seed
        # TODO: initialize W, b and training history using NumPy.

    def fit(self, X_rows: list[np.ndarray], y: np.ndarray,
            X_val_rows: list[np.ndarray] | None = None,
            y_val: np.ndarray | None = None):
        # TODO: shuffled epochs, (mini-)batch scores, softmax, loss,
        # derivatives of cross-entropy, and NumPy gradient updates.
        # Store a history of training/validation loss for plotting.
        raise NotImplementedError('Student 3: implement logistic regression fit')

    def predict_proba(self, X_rows: list[np.ndarray]) -> np.ndarray:
        # TODO: return probabilities, shape = (num_texts, num_classes).
        raise NotImplementedError('Student 3: implement predict_proba')

    def predict(self, X_rows: list[np.ndarray]) -> np.ndarray:
        # TODO: return the class with greatest predicted probability.
        raise NotImplementedError('Student 3: implement predict')

## 5. Train, validate, and plot loss

In [ ]:
# TODO: Uncomment after implementation and feature extraction.
# scratch = LogisticRegressionScratch(
#     num_features=len(bigram_vocab), num_classes=data.num_classes,
#     learning_rate=0.05, epochs=15, batch_size=64, seed=SEED,
# )
# scratch.fit(X_train_rows, data.train.labels, X_val_rows, data.val.labels)
# pred_val_custom = scratch.predict(X_val_rows)
# results_custom = sklearn_evaluation(data.val.labels, pred_val_custom, data.num_classes)
# print('Scratch LR val macro-F1:', results_custom['macro_f1'])
# TODO: plot the recorded train/validation cross-entropy losses.

## 6. Convert sparse feature indices to SciPy CSR (sklearn comparison only)
This is **not part of the scratch classifier**. scikit-learn estimators expect sparse matrices; this adapter converts your own binary bigram representation without changing the vocabulary.

In [ ]:
def sparse_rows_to_csr(rows: list[np.ndarray], num_features: int) -> csr_matrix:
    lengths = np.asarray([len(row) for row in rows], dtype=np.int64)
    indptr = np.zeros(len(rows) + 1, dtype=np.int64)
    indptr[1:] = np.cumsum(lengths)
    indices = (np.concatenate(rows).astype(np.int32, copy=False)
               if len(rows) and indptr[-1] else np.empty(0, dtype=np.int32))
    values = np.ones(len(indices), dtype=np.float64)
    return csr_matrix((values, indices, indptr), shape=(len(rows), num_features))

# TODO: Uncomment after X_train_rows / X_val_rows exist.
# X_train_sklearn = sparse_rows_to_csr(X_train_rows, len(bigram_vocab))
# X_val_sklearn = sparse_rows_to_csr(X_val_rows, len(bigram_vocab))
# print(X_train_sklearn.shape, 'nonzero:', X_train_sklearn.nnz)

## 7. Scikit-learn comparisons (validation)
Train `LogisticRegression` and `SGDClassifier` using **identical preprocessed features and splits**. The latter with `loss="log_loss"` is closest to a stochastic/minibatch cross-entropy implementation. Compare **macro-F1**, precision, recall, and confusion matrices.

In [ ]:
# TODO: Uncomment after you build X_train_sklearn and X_val_sklearn.
# sklearn_lr = LogisticRegression(max_iter=1000, random_state=SEED)
# sklearn_sgd = SGDClassifier(loss='log_loss', random_state=SEED)
# validation_rows = []
# for name, estimator in [('sklearn LogisticRegression', sklearn_lr),
#                         ('sklearn SGDClassifier', sklearn_sgd)]:
#     estimator.fit(X_train_sklearn, data.train.labels)
#     y_pred = estimator.predict(X_val_sklearn)
#     report = sklearn_evaluation(data.val.labels, y_pred, data.num_classes)
#     validation_rows.append({'model': name,
#                             'macro_precision': report['macro_precision'],
#                             'macro_recall': report['macro_recall'],
#                             'macro_f1': report['macro_f1']})
# pd.DataFrame(validation_rows)

## 8. Test evaluation — run only after choosing hyperparameters
**TODO:** Refit or retain your final model as appropriate, predict test labels and compare all three approaches. Use `sklearn_evaluation()` until Student 2 completes the from-scratch evaluator in `utils/evaluation.py`. Then validate your results with their custom metrics. Don't select hyperparameters from test performance.

In [ ]:
# TODO (final experiment only):
# X_test_sklearn = sparse_rows_to_csr(X_test_rows, len(bigram_vocab))
# y_pred_scratch = scratch.predict(X_test_rows)
# y_pred_ref_lr = sklearn_lr.predict(X_test_sklearn)
# y_pred_ref_sgd = sklearn_sgd.predict(X_test_sklearn)
# ... compare confusion matrices, per-class and macro scores.

## 9. Discussion (required)
**Binary bigram representation:**
- Advantages: TODO (e.g. local word-order information, interpretable features).
- Limitations: TODO (e.g. vocabulary size, sparsity, unseen phrases, ignoring counts).

**Classifier comparison:**
- Which approach performs better, and under what conditions? **TODO:** base this on your reported results and hyperparameters.
- Which built-in classifier is most similar to your version? **TODO:** discuss stochastic/mini-batch optimization vs standard `LogisticRegression` solvers.
- Explain any differences among implementations (convergence, regularization defaults, optimizers).